# 02. 전략 비교: `route` · `pipeline` · `compete` · `council` · `decompose`

[01. 첫 실행](01-first-run.ipynb)에서는 에이전트 하나(또는 실패 시 다음 에이전트)에게 일을 맡겼습니다.
이 노트북에서는 **여러 에이전트를 어떻게 함께 쓸지** 정하는 다섯 가지 전략을 같은 과제로 직접 돌려 보고
비교합니다.

**이런 분께 맞습니다**

- 01을 마쳤고, `run` → `diff` → `apply` 흐름이 익숙한 분
- "에이전트 여러 개를 쓰면 정말 나아지나? 비용은 얼마나 늘지?"가 궁금한 분
- 팀에서 어떤 전략을 기본값으로 둘지 정해야 하는 분

**이 노트북에서 해 보는 것**

| 장 | 하는 일 | 끝나면 알게 되는 것 |
|---|---|---|
| 0 | 전략이 왜 여러 개인지 | 전략을 고르는 기준 |
| 1 | 샌드박스와 검사(게이트) 준비 | 전략마다 결과를 같은 기준으로 판정하는 방법 |
| 2 | 다섯 전략의 dry-run 비교 | 전략별 최대 에이전트 호출 수 |
| 3 | `pipeline` | 단계별 분업과 실패 시 동작 |
| 4 | `compete` | 동시 실행과 "승자" 선택 규칙 |
| 5 | `council` | 제안 → 비평 → 종합으로 서로 배우는 과정 |
| 6 | `decompose` | 작업 나누기와 통합 방식 |
| 7 | 어떤 전략을 고를까 | 상황별 선택 가이드와 실무 팁 |

**비용과 안전**: 모든 셀이 01과 같은 샌드박스에서 **가짜 에이전트**로 실행됩니다. 실제 AI 호출과 비용은 없습니다.
소요 시간은 약 10분입니다.

## 0. 전략은 왜 여러 개인가요?

여러 에이전트를 함께 쓰는 방법은 하나가 아닙니다. 사람 팀으로 비유하면 이렇습니다.

| 전략 | 사람 팀으로 비유하면 | 결과를 고르는 방법 |
|---|---|---|
| `route` (기본값) | 담당자에게 맡기고, 못 하면 다음 사람에게 넘김 | 처음 성공한 결과 |
| `pipeline` | 한 사람이 구현하고 다음 사람이 이어서 테스트를 씀 | 마지막 단계까지 끝난 결과 |
| `compete` | 같은 과제를 여러 명에게 동시에 주고 하나를 채택 | 선택 규칙(기본: 우선순위상 처음 검사를 통과한 결과) |
| `council` | 각자 안을 내고, 서로의 안을 읽고 고친 뒤, 최종안을 냄 | 최종안 중 선택 규칙으로 하나 |
| `decompose` | 팀장이 일을 쪼개 나눠 주고, 조각이 끝나면 통합 담당이 정리 | 통합안 중 선택 규칙으로 하나 |

전략을 고를 때는 네 가지를 저울질합니다.

- **비용**: 에이전트를 몇 번 부르는가 (진짜 에이전트는 호출마다 사용량이 듭니다)
- **시간**: 동시에 돌리는가, 차례로 돌리는가
- **품질**: 한 에이전트의 판단에 기대는가, 여러 결과 중에서 고르거나 서로 다듬는가
- **작업의 모양**: 한 덩어리인가, 단계로 나뉘는가, 독립된 조각으로 쪼개지는가

어떤 전략이든 **결과는 패치로 보관되고, `apply`해야 원본에 들어간다**는 규칙은 같습니다.

## 1. 준비: 샌드박스, 과제, 검사

01과 같은 샌드박스를 만들고 `init`까지 한 번에 합니다(`init_project()`는 01의 4장 — `init` 후 설정 파일 커밋 —
을 한 줄로 묶은 도우미입니다).

과제는 모든 전략에 똑같이 줍니다.

> `textstats.py`에 단어 수를 세는 `count_words(text)`를 추가하기

그리고 01의 10장처럼 **품질 게이트**를 걸어 둡니다. 검사 스크립트는 연속된 공백(`'a b  c'` → 3)과 빈 문자열
(`''` → 0)을 확인합니다. 전략마다 "어떤 결과를 받아들일지"를 같은 기준으로 판정하기 위해서입니다.

In [1]:
from _sandbox import Sandbox

box = Sandbox.create()
box.init_project()

CHECK = (
    "from textstats import count_words\n\n"
    "assert count_words('a b  c') == 3, 'repeated spaces'\n"
    "assert count_words('') == 0, 'empty text'\n"
    "print('check_textstats: all good')\n"
)
(box.repo / "check_textstats.py").write_text(CHECK, encoding="utf-8")
(box.repo / ".gitignore").write_text("__pycache__/\n", encoding="utf-8")
toml_path = box.repo / "syncmill.toml"
toml_path.write_text(
    toml_path.read_text(encoding="utf-8").replace(
        "commands = []", 'commands = ["python3 check_textstats.py"]'
    ),
    encoding="utf-8",
)
box.git("add", "check_textstats.py", ".gitignore", "syncmill.toml")
box.git("commit", "-qm", "add a word-count check and a gate")
print("샌드박스:", box.root)

샌드박스: /tmp/syncmill-notebook.eph8jlvj


가짜 에이전트가 쓸 코드도 미리 준비합니다. 진짜 에이전트라면 각자 코드를 쓰겠지만, 여기서는 전략의 차이를
분명히 보기 위해 결과를 연출합니다.

- `GOOD` — `text.split()`: 연속 공백과 빈 문자열을 올바르게 처리 → 검사 통과
- `GOOD_DOC` — `GOOD`에 docstring을 더한 버전 → 검사 통과
- `BAD` — `text.split(' ')`: `'a b  c'`를 4개로 세고 `''`를 1개로 셈 → 검사 실패
- `TESTS` — `count_words`의 테스트 스크립트

In [2]:
GOOD = "def count_words(text):\n    return len(text.split())\n"
GOOD_DOC = (
    "def count_words(text):\n"
    '    """Count whitespace-separated words."""\n'
    "    return len(text.split())\n"
)
BAD = "def count_words(text):\n    return len(text.split(' '))\n"
TESTS = (
    "from textstats import count_words\n\n"
    "assert count_words('a b  c') == 3\n"
    "assert count_words('') == 0\n"
    "print('test_textstats: all good')\n"
)
TASK = "textstats.py에 단어 수를 세는 count_words(text)를 추가해 줘"

마지막으로 **전략이 에이전트를 어떻게 불렀는지** 보여 주는 도우미를 만듭니다. 가짜 에이전트는 syncmill이 보낸
프롬프트를 파일로 남기므로, 그것을 읽어 "누가, 어떤 단계에서, 공유 메모리 맥락을 받았는지"를 표로 보여 줍니다.

> 진짜 에이전트에는 이런 기록이 없습니다. 전략이 안에서 어떻게 움직이는지 **학습용으로 들여다보는 창**입니다.
> 단계 이름은 syncmill이 단계마다 붙이는 지시문으로 알아냅니다.

In [3]:
def show_calls(since):
    rows = box.calls(since)
    print(f"에이전트 호출 {len(rows)}번")
    print(f"{'#':>2}  {'agent':10} {'phase':22} context")
    for i, row in enumerate(rows, 1):
        print(f"{i:>2}  {row['agent']:10} {row['phase']:22} {row['context']}")
    return rows

## 2. 실행 전에 비교하기: 다섯 전략의 dry-run

전략을 바꾸기 전에 가장 먼저 할 일은 **dry-run으로 호출 수를 보는 것**입니다. 에이전트는 부르지 않으므로
비용이 들지 않습니다. 다섯 전략을 한 번에 비교해 봅시다.

In [4]:
import json

print(f"{'strategy':10} {'max calls':>9}  {'selection':9}  phases")
plans = {}
for strategy in ["route", "pipeline", "compete", "council", "decompose"]:
    out = box.sm("run", "--dry-run", TASK, "-s", strategy, show=False).stdout
    plan = json.loads(out[: out.index("\n}\n") + 2])
    plans[strategy] = plan
    print(f"{strategy:10} {plan['maximum_agent_calls']:>9}  {plan['selection']:9}  {' → '.join(plan['phases'])}")

strategy   max calls  selection  phases


route              2  n/a        route


pipeline           2  n/a        pipeline


compete            2  verified   compete → select


council            6  verified   propose → critique-1 → synthesize → select


decompose         11  verified   plan → subtasks → synthesize → select


**출력 읽는 법** — 에이전트 2개(`codex`, `kimi-code`) 기준입니다.

| 전략 | 최대 호출 | 왜 그 숫자인가 |
|---|---|---|
| `route` | 2 | 하나씩 시도, 성공하면 멈춤 → 보통은 1번 |
| `pipeline` | 2 | 에이전트마다 한 단계씩 |
| `compete` | 2 | 모두 동시에 한 번씩 |
| `council` | 6 | 에이전트마다 제안 1 + 비평 1(라운드 수만큼) + 종합 1 |
| `decompose` | 11 | 계획 1 + 하위 작업 최대 8 + 종합 2(에이전트 수) |

- `phases`는 전략이 거치는 단계입니다. `select`는 여러 후보 중 하나를 고르는 단계입니다.
- `selection`의 `verified`는 기본 선택 규칙(4장에서 설명), `n/a`는 고를 후보가 하나뿐이라 해당 없음입니다.
- 이 숫자는 **에이전트 호출 횟수의 상한**이지 비용의 상한이 아닙니다. 실제 비용은 각 호출이 쓰는 모델과 토큰 양에 따라 달라집니다.
- 또 이 숫자는 최대값입니다. `route`는 첫 에이전트가 성공하면 1번으로 끝나고, `decompose`는 계획이 몇 조각이냐에
  따라 달라집니다(6장에서 실제 호출 수와 비교합니다).
- 에이전트를 3개 쓰면 `council`은 9번, `compete`는 3번처럼 **에이전트 수에 비례해** 늘어납니다.

## 3. `pipeline`: 단계별 분업

**언제 쓰나요?** "구현 → 테스트 작성 → 문서화"처럼 **앞 단계의 결과 위에 다음 단계를 쌓는** 작업에 맞습니다.

**어떻게 동작하나요?**

- 에이전트들이 우선순위 순서대로 **한 작업 공간을 이어받아** 차례로 일합니다. 뒤 단계는 앞 단계가 만든 파일을
  그대로 봅니다.
- 앞 단계의 답변은 공유 메모리(`pipeline-stage` 태그)를 통해 뒤 단계 프롬프트 앞에도 붙습니다.
- 품질 게이트는 **마지막 단계가 끝난 결과에만** 적용됩니다. 중간 단계는 아직 미완성일 수 있기 때문입니다.

이번 연출에서 codex(1단계)는 `count_words`를 구현하고, kimi-code(2단계)는 테스트 파일을 씁니다.

In [5]:
box.set_scenario({
    "codex": {"files": {"textstats.py": GOOD}, "message": "Added count_words."},
    "kimi-code": {"files": {"test_textstats.py": TESTS}, "message": "Added tests for count_words."},
})

start = len(box.prompts())
piped = box.sm("run", TASK + " 그리고 테스트도 작성해 줘", "-s", "pipeline")
piped_id = box.run_id(piped)
pipeline_calls = show_calls(start)
assert [c["agent"] for c in pipeline_calls] == ["codex", "kimi-code"]

[syncmill] Executing task be2ab6d0...
[syncmill] Agents: codex → kimi-code (pipeline)

[kimi-code] Completed in 0.68s
[syncmill] Quality gate: passed (1/1 checks passed)
[syncmill] Files changed: test_textstats.py, textstats.py

Added tests for count_words.
[syncmill] run: b88f94a3-1923-493a-9c49-28bc1f361c0a
[syncmill] artifact: /private/tmp/syncmill-notebook.eph8jlvj/state/runs/2f1c63be-3280-42ad-9446-2787906ee74b/runs/b88f94a3-1923-493a-9c49-28bc1f361c0a
[syncmill] review: syncmill diff b88f94a3-1923-493a-9c49-28bc1f361c0a
[syncmill] apply: syncmill apply b88f94a3-1923-493a-9c49-28bc1f361c0a
에이전트 호출 2번
 #  agent      phase                  context
 1  codex      task                   no
 2  kimi-code  task                   yes


**출력 읽는 법**

- `Agents: codex → kimi-code (pipeline)` — 이 순서로 단계를 잇습니다.
- `[kimi-code] Completed ...` — 기본 출력에는 **마지막 단계만** 나옵니다. 1단계(codex)도 실행됐다는 것은 호출 표로
  확인할 수 있습니다.
- `Quality gate: passed` — 두 단계가 모두 끝난 최종 결과가 검사를 통과했습니다.
- `Files changed: test_textstats.py, textstats.py` — 두 단계의 작업이 **한 결과에 함께** 담겼습니다.
- 호출 표의 `context`: 1단계는 `no`(앞선 기록 없음), 2단계는 `yes` — 1단계의 답변이 붙어 있습니다.

2단계가 받은 프롬프트의 앞부분을 보면, 1단계 codex의 답변이 들어 있습니다.

In [6]:
stage2 = box.prompts()[-1].read_text(encoding="utf-8")
print(stage2[: stage2.index("Task: ")])
assert "Added count_words." in stage2

Context from previous work:
[/private/tmp/syncmill-notebook.eph8jlvj/state/memory/files/2026-] **Task Result**

**Prompt:** textstats.py에 단어 수를 세는 count_words(text)를 추가해 줘 그리고 테스트도 작성해 줘

**Agent:** codex

**Output:**
Added count_words.

---




diff에도 두 파일이 모두 있습니다.

In [7]:
patch = box.sm("diff", piped_id).stdout
changed = [line[len("+++ b/"):] for line in patch.splitlines() if line.startswith("+++ b/")]
print("diff에 담긴 파일:", changed)
assert sorted(changed) == ["test_textstats.py", "textstats.py"]

diff --git a/test_textstats.py b/test_textstats.py
new file mode 100644
index 0000000..5dbd77e
--- /dev/null
+++ b/test_textstats.py
@@ -0,0 +1,5 @@
+from textstats import count_words
+
+assert count_words('a b  c') == 3
+assert count_words('') == 0
+print('test_textstats: all good')
diff --git a/textstats.py b/textstats.py
new file mode 100644
index 0000000..9a68e44
--- /dev/null
+++ b/textstats.py
@@ -0,0 +1,2 @@
+def count_words(text):
+    return len(text.split())
diff에 담긴 파일: ['test_textstats.py', 'textstats.py']


**단계가 실패하면?** 파이프라인은 실패한 단계에서 **멈춥니다**(fail-fast). 2단계(kimi-code)를 일부러 실패시켜 봅시다.

In [8]:
start = len(box.prompts())
broken = box.sm("run", TASK, "-s", "pipeline", check=False, extra_env={"FAKE_KIMI_FAIL": "1"})
assert broken.returncode == 1
broken_calls = show_calls(start)

[syncmill] Executing task ee41d8f8...
[syncmill] Agents: codex → kimi-code (pipeline)

[syncmill] Failed
Pipeline failed at stage 2/2 (kimi-code): kimi (fake): simulated failure
[syncmill] run: 5d48590c-121f-47a3-ba9b-cdb04288e008
[syncmill] artifact: /private/tmp/syncmill-notebook.eph8jlvj/state/runs/2f1c63be-3280-42ad-9446-2787906ee74b/runs/5d48590c-121f-47a3-ba9b-cdb04288e008
(exit code 1)
에이전트 호출 2번
 #  agent      phase                  context
 1  codex      task                   yes
 2  kimi-code  task                   yes


- `Pipeline failed at stage 2/2 (kimi-code): ...` — 몇 번째 단계에서, 어느 에이전트가, 왜 실패했는지 알려 줍니다.
- 1단계는 성공했지만 **결과 전체가 실패**로 처리되고 적용할 패치도 남지 않습니다. 중간까지 된 작업만 따로 받을 수는
  없습니다.
- `runs list`에서 이 실패의 에이전트 열에는 실패한 단계의 에이전트(`kimi-code`)가 나옵니다.

**실무 팁**

- 단계 순서는 에이전트 우선순위(`agent_priority`) 순서입니다. "누가 무엇을 맡을지"는 요청 문장에 단계를 분명히
  적어 주는 편이 좋습니다(예: "구현하고, 테스트를 추가하고, README에 사용법을 적어 줘").
- 단계가 많을수록 한 단계의 실패로 전체를 다시 돌릴 위험이 커집니다. 2~3단계로 시작하세요.

## 4. `compete`: 동시에 시키고 하나 고르기

**언제 쓰나요?** 같은 과제를 여러 에이전트에게 **동시에** 시켜, 한 에이전트가 틀리더라도 다른 에이전트의 맞는 결과를
받고 싶을 때입니다.

**어떻게 동작하나요?**

- 모든 에이전트가 **각자의 작업 공간에서 동시에** 같은 과제를 풉니다. 서로의 작업은 보지 못합니다.
- 끝나면 **선택 규칙**으로 결과 하나를 고릅니다. 기본 규칙 `verified`는 "우선순위 순서대로 보면서 **처음으로
  게이트를 통과한** 결과"입니다. 빨리 끝난 순서가 아닙니다.

### 4-1. 한 에이전트가 틀렸을 때

codex는 `BAD`(검사 실패), kimi-code는 `GOOD`(검사 통과)를 내도록 연출합니다.

In [9]:
box.set_scenario({
    "codex": {"files": {"textstats.py": BAD}, "message": "Split on single spaces."},
    "kimi-code": {"files": {"textstats.py": GOOD}, "message": "Split on any whitespace."},
})
start = len(box.prompts())
won = box.sm("run", TASK, "-s", "compete")
show_calls(start)
won_manifest = json.loads(box.sm("runs", "show", box.run_id(won), show=False).stdout)
print("채택된 에이전트:", won_manifest["agent_id"])
assert won_manifest["agent_id"] == "kimi-code"

[syncmill] Executing task c2150a3d...
[syncmill] Agents: codex → kimi-code (compete)

[kimi-code] Completed in 0.02s
[syncmill] Quality gate: passed (1/1 checks passed)
[syncmill] Files changed: textstats.py

Split on any whitespace.
[syncmill] run: 8f6b3587-e988-4ebe-bae0-45cf09d00027
[syncmill] artifact: /private/tmp/syncmill-notebook.eph8jlvj/state/runs/2f1c63be-3280-42ad-9446-2787906ee74b/runs/8f6b3587-e988-4ebe-bae0-45cf09d00027
[syncmill] review: syncmill diff 8f6b3587-e988-4ebe-bae0-45cf09d00027
[syncmill] apply: syncmill apply 8f6b3587-e988-4ebe-bae0-45cf09d00027
에이전트 호출 2번
 #  agent      phase                  context
 1  codex      task                   yes
 2  kimi-code  task                   yes


채택된 에이전트: kimi-code


- 두 에이전트가 모두 호출됐지만(호출 표), 결과는 **검사를 통과한 kimi-code**가 채택됐습니다.
- 우선순위가 더 높은 codex의 결과는 게이트에서 탈락했습니다. 01의 `route`와 달리 codex가 끝나기를 기다렸다가
  kimi-code를 부르는 것이 아니라, **처음부터 둘을 함께** 돌렸다는 점이 다릅니다.
- 기본 출력에는 채택된 결과만 나옵니다.

### 4-2. 둘 다 통과할 코드를 냈을 때

이번에는 둘 다 검사를 통과할 코드를 내되, kimi-code 쪽에만 docstring이 있습니다. 사람이 보기엔 kimi-code 쪽이 조금 더
좋아 보입니다. 어느 쪽이 채택될까요?

In [10]:
box.set_scenario({
    "codex": {"files": {"textstats.py": GOOD}, "message": "codex version"},
    "kimi-code": {"files": {"textstats.py": GOOD_DOC}, "message": "kimi-code version with a docstring"},
})
tie = box.sm("run", TASK, "-s", "compete")
tie_manifest = json.loads(box.sm("runs", "show", box.run_id(tie), show=False).stdout)
print("채택된 에이전트:", tie_manifest["agent_id"])
assert tie_manifest["agent_id"] == "codex"

[syncmill] Executing task 2d845a0d...
[syncmill] Agents: codex → kimi-code (compete)

[codex] Completed in 0.03s
[syncmill] Quality gate: passed (1/1 checks passed)
[syncmill] Files changed: textstats.py

codex version
[syncmill] run: e5032b1d-ff1a-42fb-b047-0c34f0b45aee
[syncmill] artifact: /private/tmp/syncmill-notebook.eph8jlvj/state/runs/2f1c63be-3280-42ad-9446-2787906ee74b/runs/e5032b1d-ff1a-42fb-b047-0c34f0b45aee
[syncmill] review: syncmill diff e5032b1d-ff1a-42fb-b047-0c34f0b45aee
[syncmill] apply: syncmill apply e5032b1d-ff1a-42fb-b047-0c34f0b45aee


채택된 에이전트: codex


**codex**가 채택됐습니다. 기본 규칙 `verified`는 결과의 **품질을 비교하지 않습니다.** 우선순위 순서대로 검사하다가
처음 통과한 결과에서 **바로 멈춥니다.** 그래서 codex가 통과하자 kimi-code의 결과는 검사조차 하지 않았습니다. 둘 다 통과할 수 있는
결과라면 언제나 우선순위가 높은 쪽이 이깁니다.

그래서 실무에서는 **게이트가 곧 품질 기준**입니다. "docstring이 있어야 한다"가 중요하면 그것을 검사 명령으로
만들어야 합니다. 결과끼리 비교해 순위를 매기게 하고 싶다면 `--selection judge`를 씁니다. 에이전트 하나가 후보들을
비교 판정하므로 **호출이 한 번 더** 듭니다(판정에 실패하면 `verified`로 돌아갑니다).

In [11]:
judge_plan = box.sm("run", "--dry-run", TASK, "-s", "compete", "--selection", "judge", show=False).stdout
judge_plan = json.loads(judge_plan[: judge_plan.index("\n}\n") + 2])
print("selection:", judge_plan["selection"], "/ 최대 호출:", judge_plan["maximum_agent_calls"])
assert judge_plan["maximum_agent_calls"] == plans["compete"]["maximum_agent_calls"] + 1

selection: judge / 최대 호출: 3


**실무 팁**

- 비용은 **에이전트 수만큼** 듭니다. 한 에이전트가 자주 틀리는 어려운 과제, 또는 실패하면 안 되는 과제에 쓰세요.
- 게이트 없이 `compete`를 쓰면 대개 우선순위 1번 에이전트의 결과가 채택됩니다. **게이트와 함께 써야** 의미가 큽니다.
- 터미널(tmux)에서 `--monitor`를 붙이면 에이전트별 진행 상황을 창을 나눠 실시간으로 볼 수 있습니다
  (`compete`·`council`·`decompose` 같은 동시 실행 전략에서 동작).

## 5. `council`: 서로의 안을 읽고 다듬기

**언제 쓰나요?** 정답이 하나로 정해지지 않는 설계 문제나, 한 에이전트가 놓치기 쉬운 부분이 있는 어려운 과제에
맞습니다. "여러 명이 각자 안을 내고, 서로 검토한 뒤 최종안을 낸다"는 방식입니다.

**어떻게 동작하나요?** 세 단계를 거칩니다(모두 에이전트들이 동시에 실행).

1. **제안(propose)** — 각자 자기 작업 공간에서 해결책을 만들고, 답변을 공유 메모리에 올립니다.
2. **비평(critique)** — 다른 에이전트들의 제안을 프롬프트 맥락으로 받아 읽고, **자기 작업 공간의 해결책을** 다듬습니다.
   설정한 라운드 수(기본 1)만큼 반복합니다.
3. **종합(synthesize)** — 지금까지의 논의를 바탕으로 각자 최종안을 내고, `compete`와 같은 선택 규칙으로 하나를 고릅니다.
   게이트는 이 최종안에만 적용됩니다.

에이전트끼리 직접 대화하지는 않습니다. 서로의 안은 **공유 메모리를 통해서만** 전달됩니다.

이번 연출: codex는 처음에 `BAD`를 제안했다가, kimi-code의 제안(`split()`이 연속 공백을 처리한다)을 읽은 뒤
`GOOD`으로 고치고, 최종안에서는 docstring까지 붙입니다. kimi-code는 처음부터 `GOOD`입니다.

In [12]:
box.set_scenario({
    "codex": [
        {"when": "Draft and implement your best solution",
         "files": {"textstats.py": BAD}, "message": "Proposal: split(' ')."},
        {"when": "Critique those ideas",
         "files": {"textstats.py": GOOD}, "message": "Adopted kimi-code's split(); it handles repeated spaces."},
        {"when": "Synthesize them into a single best FINAL solution",
         "files": {"textstats.py": GOOD_DOC}, "message": "Final: split() with a docstring."},
    ],
    "kimi-code": [
        {"files": {"textstats.py": GOOD}, "message": "Proposal: split() handles repeated spaces."},
    ],
})
start = len(box.prompts())
council = box.sm("run", TASK, "-s", "council")
council_id = box.run_id(council)
council_calls = show_calls(start)
assert len(council_calls) == plans["council"]["maximum_agent_calls"]

[syncmill] Executing task cf7287ef...
[syncmill] Agents: codex → kimi-code (council)

[codex] Completed in 0.09s
[syncmill] Quality gate: passed (1/1 checks passed)
[syncmill] Files changed: textstats.py

Final: split() with a docstring.
[syncmill] run: 48a32f1f-3b4d-4799-8041-e0330eb18110
[syncmill] artifact: /private/tmp/syncmill-notebook.eph8jlvj/state/runs/2f1c63be-3280-42ad-9446-2787906ee74b/runs/48a32f1f-3b4d-4799-8041-e0330eb18110
[syncmill] review: syncmill diff 48a32f1f-3b4d-4799-8041-e0330eb18110
[syncmill] apply: syncmill apply 48a32f1f-3b4d-4799-8041-e0330eb18110
에이전트 호출 6번
 #  agent      phase                  context
 1  codex      council: propose       yes
 2  kimi-code  council: propose       yes
 3  codex      council: critique      yes
 4  kimi-code  council: critique      yes
 5  codex      council: synthesize    yes
 6  kimi-code  council: synthesize    yes


**출력 읽는 법**

- 호출 표: 에이전트마다 `propose` → `critique` → `synthesize` 세 번, 모두 6번 호출됐습니다. dry-run의 최대값(6)과 같습니다.
  `council`은 단계를 건너뛰지 않으므로 보통 최대값만큼 호출합니다.
- 같은 단계 안에서는 에이전트들이 **동시에** 실행되므로, 표에서 같은 단계 안의 순서(codex가 먼저인지 kimi-code가
  먼저인지)는 실행할 때마다 달라질 수 있습니다. 단계의 순서는 항상 같습니다.
- `[codex] Completed ...`, `Final: split() with a docstring.` — 우선순위 1번인 codex의 최종안이 검사를 통과해 바로 채택됐습니다
  (`verified` 규칙이므로 kimi-code의 최종안은 검사하지 않았습니다).
- 출력에는 최종 결과만 나옵니다. 제안·비평 단계의 결과는 패치가 아니라 **공유 메모리의 기록**으로 남습니다
  (`council-proposal`, `council-critique` 태그).

codex가 **비평 단계에서 받은 프롬프트**에 kimi-code의 제안이 들어 있었는지 확인해 봅시다.

In [13]:
critique = next(
    p.read_text(encoding="utf-8") for p in box.prompts()[start:]
    if p.name.startswith("codex-") and "Critique those ideas" in p.read_text(encoding="utf-8")
)
print("kimi-code의 제안이 맥락에 있나요?", "Proposal: split() handles repeated spaces." in critique)
assert "Proposal: split() handles repeated spaces." in critique

patch = box.sm("diff", council_id).stdout
assert '+    """Count whitespace-separated words."""' in patch.splitlines()

kimi-code의 제안이 맥락에 있나요? True


diff --git a/textstats.py b/textstats.py
new file mode 100644
index 0000000..8bb4ed0
--- /dev/null
+++ b/textstats.py
@@ -0,0 +1,3 @@
+def count_words(text):
+    """Count whitespace-separated words."""
+    return len(text.split())


codex는 kimi-code와 직접 이야기하지 않았지만, 비평 단계 프롬프트에 kimi-code의 제안이 실려 왔고, 최종 결과는
처음 제안(`BAD`)보다 나아졌습니다. 이것이 `council`이 노리는 효과입니다.

**주의할 점**

- 호출 수가 **에이전트 수와 라운드 수 모두에 비례해** 늘어납니다. 에이전트당 `2 + 라운드 수`번이므로, 에이전트 3개에 라운드 2면 12번입니다.
  라운드 수는 `syncmill.toml`의 `[council]` `rounds`(또는 `SYNCMILL_COUNCIL__ROUNDS`)로 정합니다.
- 서로의 안은 공유 메모리의 **관련도 검색**으로 전달됩니다. 모든 제안이 빠짐없이 전달된다는 보장은 없습니다.
- 최종 선택 규칙은 `compete`와 같습니다. 여기서도 게이트가 품질 기준입니다.

## 6. `decompose`: 나눠서 맡기고 통합하기

**언제 쓰나요?** 서로 독립적인 조각으로 깔끔하게 나뉘는 큰 작업("API 엔드포인트 세 개 추가", "모듈 구현 + 테스트 +
문서")에 맞습니다.

**어떻게 동작하나요?**

1. **계획(plan)** — 우선순위 1번 에이전트가 작업을 조각(하위 작업) 목록으로 나눕니다. 결과는 JSON 배열이어야 하며,
   형식이 틀리면 **작업 전체를 하나의 조각으로** 대신 진행합니다(계획 실패로 멈추지 않음).
2. **하위 작업(subtasks)** — 조각을 에이전트들에게 **돌아가며** 나눠 주고, 각자의 작업 공간에서 동시에 처리합니다.
   각 조각의 답변(요약)은 공유 메모리에 올라갑니다.
3. **종합(synthesize)** — 모든 에이전트가 **새 작업 공간**에서 조각들의 요약을 읽고 전체 구현을 다시 작성합니다.
   그중 하나를 선택 규칙으로 고르고, 게이트는 이 종합 결과에만 적용됩니다.

중요한 점: 하위 작업의 **코드를 그대로 합치지 않습니다.** 조각 작업 공간은 버려지고, 종합 단계가 요약을 바탕으로 다시
씁니다. 그래서 종합을 맡는 에이전트가 조각들의 결과를 제대로 반영하는지가 품질을 좌우합니다.

이번 연출: codex가 "함수 추가", "테스트 추가" 두 조각으로 계획을 세우고, 조각 1은 codex, 조각 2는 kimi-code가 맡습니다.
종합에서는 둘 다 함수와 테스트를 함께 씁니다.

In [14]:
PLAN = ["textstats.py에 count_words 함수 추가", "count_words 테스트 스크립트 test_textstats.py 추가"]
box.set_scenario({
    "codex": [
        {"when": "You are the planner", "files": {}, "message": json.dumps(PLAN, ensure_ascii=False)},
        {"when": "YOUR subtask to implement now:", "files": {"textstats.py": GOOD}, "message": "Implemented count_words."},
        {"when": "Integrate that work", "files": {"textstats.py": GOOD, "test_textstats.py": TESTS},
         "message": "Integrated the function and its tests."},
    ],
    "kimi-code": [
        {"when": "YOUR subtask to implement now:", "files": {"test_textstats.py": TESTS}, "message": "Wrote tests."},
        {"when": "Integrate that work", "files": {"textstats.py": GOOD_DOC, "test_textstats.py": TESTS},
         "message": "Integrated with a docstring."},
    ],
})
start = len(box.prompts())
split = box.sm("run", TASK + " 그리고 테스트도 작성해 줘", "-s", "decompose")
split_id = box.run_id(split)
split_calls = show_calls(start)

[syncmill] Executing task 17349d20...
[syncmill] Agents: codex → kimi-code (decompose)

[codex] Completed in 0.09s
[syncmill] Quality gate: passed (1/1 checks passed)
[syncmill] Files changed: test_textstats.py, textstats.py

Integrated the function and its tests.
[syncmill] run: 85376cd7-61b1-4b30-aaef-349738e04dbe
[syncmill] artifact: /private/tmp/syncmill-notebook.eph8jlvj/state/runs/2f1c63be-3280-42ad-9446-2787906ee74b/runs/85376cd7-61b1-4b30-aaef-349738e04dbe
[syncmill] review: syncmill diff 85376cd7-61b1-4b30-aaef-349738e04dbe
[syncmill] apply: syncmill apply 85376cd7-61b1-4b30-aaef-349738e04dbe
에이전트 호출 5번
 #  agent      phase                  context
 1  codex      decompose: plan        yes
 2  codex      decompose: subtask     yes
 3  kimi-code  decompose: subtask     yes
 4  codex      decompose: synthesize  yes
 5  kimi-code  decompose: synthesize  yes


**출력 읽는 법**

- 호출 표: `plan` 1번(codex) → `subtask` 2번(조각 1은 codex, 조각 2는 kimi-code — 돌아가며 배정) → `synthesize` 2번.
- 실제 호출은 **5번**으로, dry-run의 최대값(11번)보다 적습니다. 계획이 2조각이었기 때문입니다. 최대값은 조각이 상한(8개)까지
  나올 때를 가정한 숫자입니다.
- `[codex] Completed ...`, `Integrated the function and its tests.` — 종합 결과 중 codex의 것이 채택됐습니다.

각 하위 작업이 받은 조각 지시를 보면, 계획이 그대로 나뉘어 배정된 것을 알 수 있습니다.

In [15]:
for row in split_calls:
    if row["phase"] == "decompose: subtask":
        print(f"{row['agent']:10} → {row['task'].rpartition('YOUR subtask to implement now: ')[2]}")

assert [r["phase"] for r in split_calls] == (
    ["decompose: plan"] + ["decompose: subtask"] * 2 + ["decompose: synthesize"] * 2
)
assert len(split_calls) < plans["decompose"]["maximum_agent_calls"]

codex      → textstats.py에 count_words 함수 추가
kimi-code  → count_words 테스트 스크립트 test_textstats.py 추가


최종 diff는 **종합 단계 codex의 작업 공간**에서 나온 것입니다. 조각 작업 공간의 코드를 합친 것이 아닙니다.

In [16]:
patch = box.sm("diff", split_id).stdout
changed = [line[len("+++ b/"):] for line in patch.splitlines() if line.startswith("+++ b/")]
print("diff에 담긴 파일:", changed)
assert sorted(changed) == ["test_textstats.py", "textstats.py"]

diff --git a/test_textstats.py b/test_textstats.py
new file mode 100644
index 0000000..5dbd77e
--- /dev/null
+++ b/test_textstats.py
@@ -0,0 +1,5 @@
+from textstats import count_words
+
+assert count_words('a b  c') == 3
+assert count_words('') == 0
+print('test_textstats: all good')
diff --git a/textstats.py b/textstats.py
new file mode 100644
index 0000000..9a68e44
--- /dev/null
+++ b/textstats.py
@@ -0,0 +1,2 @@
+def count_words(text):
+    return len(text.split())
diff에 담긴 파일: ['test_textstats.py', 'textstats.py']


**주의할 점**

- **계획의 품질이 전체를 좌우합니다.** 조각이 서로 의존하면(예: "함수 추가"와 "그 함수를 쓰는 코드") 각 조각은 다른 조각을
  볼 수 없어 어긋나기 쉽습니다. 서로 독립적인 조각으로 나뉘는 작업에 쓰세요.
- 조각 수의 상한은 `[decompose]` `max_subtasks`(기본 8, `SYNCMILL_DECOMPOSE__MAX_SUBTASKS`)입니다. 비용을 묶어 두고
  싶으면 줄이세요.
- 하위 작업은 게이트 검사를 받지 않습니다(조각 하나만으로는 완성품이 아니므로). 최종 품질은 종합 결과의 게이트가 지킵니다.

## 7. 어떤 전략을 고를까?

| 상황 | 추천 | 이유 |
|---|---|---|
| 처음 쓰거나, 작고 분명한 과제 | `route` (또는 `--agent` 하나) | 가장 싸고 단순. 실패하면 다음 에이전트로 |
| 구현 → 테스트 → 문서처럼 단계가 분명함 | `pipeline` | 앞 단계 결과 위에 이어서 쌓음 |
| 틀리면 곤란하고, 검사로 정답을 판정할 수 있음 | `compete` + 게이트 | 여러 결과 중 검사를 통과한 것을 받음 |
| 정답이 하나가 아닌 설계 문제 | `council` | 서로의 안을 읽고 다듬음. 호출 수가 많음(에이전트당 2 + 라운드 수) |
| 독립적인 조각으로 나뉘는 큰 작업 | `decompose` | 조각을 동시에 처리한 뒤 통합 |

**실무 팁**

1. **`route`로 시작하세요.** 대부분의 일상 과제는 이것으로 충분합니다.
2. **동시 전략 전에 게이트부터.** `compete`·`council`·`decompose`의 선택 규칙은 품질을 비교하지 않습니다. 게이트가
   없으면 "여러 번 돌리고 대개 1번 에이전트 결과를 받는" 것이 되기 쉽습니다.
3. **새 전략은 dry-run부터.** 최대 호출 수를 확인하세요(호출 횟수의 상한이며, 실제 비용은 모델·토큰 양에 따라 다름).
4. **팀 기본값은 설정 파일로.** `syncmill.toml`의 `default_strategy = "route"`를 바꾸면 `-s` 없이 실행할 때의 전략이
   바뀝니다. 자주 쓰는 에이전트 조합·전략은 팀 프로필(`[teams.<이름>]`, `--team`)로 저장할 수도 있습니다.
5. **어떤 전략이든 마지막은 `diff` → `apply`.** 전략은 "결과를 어떻게 만들고 고르는가"만 바꿀 뿐, 원본에 반영하는 것은
   언제나 여러분입니다.

전략별 자세한 옵션은 [사용 안내서의 "전략 고르기"](../user-guide.ko.md#전략-고르기)를 보세요.

## 8. 정리하기

이번 노트북에서 만든 결과는 하나도 적용하지 않았습니다(모두 `pending` 또는 `failed`로 남아 있습니다). 샌드박스를 지웁니다.

In [17]:
box.set_scenario(None)
box.sm("runs", "list")
root = box.root
box.cleanup()
print("삭제됨:", not root.exists())

85376cd7-61b1-4b30-aaef-349738e04dbe  pending      codex      2026-10-14T11:55:54.301714+00:00
48a32f1f-3b4d-4799-8041-e0330eb18110  pending      codex      2026-10-14T11:55:52.814789+00:00
e5032b1d-ff1a-42fb-b047-0c34f0b45aee  pending      codex      2026-10-14T11:55:50.880471+00:00
8f6b3587-e988-4ebe-bae0-45cf09d00027  pending      kimi-code  2026-10-14T11:55:49.418506+00:00
5d48590c-121f-47a3-ba9b-cdb04288e008  failed       kimi-code  2026-10-14T11:55:48.294522+00:00
b88f94a3-1923-493a-9c49-28bc1f361c0a  pending      kimi-code  2026-10-14T11:55:45.910693+00:00
삭제됨: True


## 정리: 이번에 쓴 명령

| 명령 | 뜻 |
|---|---|
| `syncmill strategies` | 사용할 수 있는 전략 목록 |
| `syncmill run --dry-run "<작업>" -s <전략>` | 전략별 단계와 최대 호출 수 확인 |
| `syncmill run "<작업>" -s pipeline` | 단계별 분업 |
| `syncmill run "<작업>" -s compete` | 동시에 시키고 선택 규칙으로 하나 채택 |
| `syncmill run "<작업>" -s compete --selection judge` | 결과를 비교 판정해서 채택(호출 1번 추가) |
| `syncmill run "<작업>" -s council` | 제안 → 비평 → 종합 |
| `syncmill run "<작업>" -s decompose` | 계획 → 하위 작업 → 종합 |
| `syncmill run "<작업>" -s compete --monitor` | 동시 실행을 tmux 창으로 실시간 보기 |

다음 노트북: **03. 공유 메모리** — 이번에 계속 등장한 "Context from previous work"가 어디에 저장되고 어떻게 고르는지
들여다봅니다.